# TPCH Business SQL Queries

In [0]:
Select
    Round(Sum(extended_price * (1-discount_rate)), 2) as net_revenue,
    count(distinct order_key) as total_orders,
    count(distinct customer_key) as active_customers,
    Round(Sum(extended_price * (1-discount_rate)) / count(Distinct order_key), 2) as average_order_value
From tpch_dev.gold.fact_sales

-- 

In [0]:
Select
    Year(order_date) as Year,
    Month(order_date) as Month,
    Round(Sum(extended_price * (1-discount_rate)), 2) as net_revenue
From tpch_dev.gold.fact_sales
Group by Year, Month
order by Year, Month

In [0]:
WITH annual_revenue AS (
    SELECT
        YEAR(order_date) AS order_year,
        ROUND(SUM(extended_price * (1 - discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales
    GROUP BY YEAR(order_date)
)
SELECT
    order_year,
    net_revenue,
    LAG(net_revenue) OVER (ORDER BY order_year)                        AS prev_year_revenue,
    ROUND(
        (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year))
        / LAG(net_revenue) OVER (ORDER BY order_year) * 100,
        2
    )                                                              AS yoy_change_pct
FROM annual_revenue
ORDER BY order_year

In [0]:
SELECT
    c.customer_market_segment,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
GROUP BY c.customer_market_segment
ORDER BY net_revenue DESC

In [0]:
SELECT
    s.customer_key,
    c.customer_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
GROUP BY s.customer_key, c.customer_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
WITH customer_revenue AS (
    SELECT
        s.customer_key,
        c.customer_name,
        ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales AS s
    JOIN tpch_dev.gold.dim_customer AS c
        ON s.customer_key = c.customer_key
    GROUP BY s.customer_key, c.customer_name
),
top_10 AS (
    SELECT *
    FROM customer_revenue
    ORDER BY net_revenue DESC
    LIMIT 10
)
SELECT
    customer_key,
    customer_name,
    net_revenue,
    ROUND(net_revenue / SUM(net_revenue) OVER () * 100, 2) AS pct_of_total_revenue
FROM top_10
ORDER BY net_revenue DESC

In [0]:
SELECT
    s.product_key,
    p.product_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_product AS p
    ON s.product_key = p.product_key
GROUP BY s.product_key, p.product_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
SELECT
    s.product_key,
    p.product_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_product AS p
    ON s.product_key = p.product_key
GROUP BY s.product_key, p.product_name
ORDER BY net_revenue ASC
LIMIT 10

In [0]:
SELECT
    p.product_type,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_product AS p
    ON s.product_key = p.product_key
GROUP BY p.product_type
ORDER BY net_revenue DESC

In [0]:
SELECT
    s.supplier_key,
    sp.supplier_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_supplier AS sp
    ON s.supplier_key = sp.supplier_key
GROUP BY s.supplier_key, sp.supplier_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
WITH supplier_revenue AS (
    SELECT
        s.supplier_key,
        sp.supplier_name,
        ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales AS s
    JOIN tpch_dev.gold.dim_supplier AS sp
        ON s.supplier_key = sp.supplier_key
    GROUP BY s.supplier_key, sp.supplier_name
),
top_10 AS (
    SELECT *
    FROM supplier_revenue
    ORDER BY net_revenue DESC
    LIMIT 10
)
SELECT
    supplier_key,
    supplier_name,
    net_revenue,
    ROUND(net_revenue / SUM(net_revenue) OVER () * 100, 2) AS pct_of_top10_revenue
FROM top_10
ORDER BY net_revenue DESC


In [0]:
SELECT
    g.region_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
JOIN tpch_dev.gold.dim_geography AS g
    ON c.nation_key = g.geography_key
GROUP BY g.region_name
ORDER BY net_revenue DESC

In [0]:
SELECT
    g.nation_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
JOIN tpch_dev.gold.dim_geography AS g
    ON c.nation_key = g.geography_key
GROUP BY g.nation_name
ORDER BY net_revenue DESC

In [0]:
SELECT
    s.return_flag,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT s.order_key) AS total_orders
FROM tpch_dev.gold.fact_sales AS s
GROUP BY s.return_flag
ORDER BY net_revenue DESC

In [0]:
SELECT
    c.customer_market_segment,
    ROUND(AVG(s.discount_rate) * 100, 2) AS avg_discount_pct
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
GROUP BY c.customer_market_segment
ORDER BY avg_discount_pct DESC

In [0]:
SELECT
    ship_mode,
    ROUND(AVG(DATEDIFF(receipt_date, ship_date)), 2) AS avg_ship_to_receipt_days
FROM tpch_dev.gold.fact_sales
GROUP BY ship_mode
ORDER BY avg_ship_to_receipt_days DESC

In [0]:
SELECT
    order_ship_priority,
    COUNT(DISTINCT order_key) AS total_orders,
    ROUND(SUM(extended_price * (1 - discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales
GROUP BY order_ship_priority
ORDER BY net_revenue DESC

In [0]:
SELECT
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month,
    order_status,
    COUNT(DISTINCT order_key) AS total_orders,
    ROUND(SUM(extended_price * (1 - discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales
GROUP By order_year, order_month, order_status
ORDER BY order_year, order_month, total_orders DESC

In [0]:
WITH product_segment_revenue AS (
    SELECT
        c.customer_market_segment,
        s.product_key,
        p.product_name,
        ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales AS s
    JOIN tpch_dev.gold.dim_product AS p
        ON s.product_key = p.product_key
    JOIN tpch_dev.gold.dim_customer AS c
        ON s.customer_key = c.customer_key
    GROUP BY c.customer_market_segment, s.product_key, p.product_name
)
SELECT
    customer_market_segment,
    product_key,
    product_name,
    net_revenue,
    RANK() OVER (
        PARTITION BY customer_market_segment
        ORDER BY net_revenue DESC
    ) AS revenue_rank
FROM product_segment_revenue
ORDER BY customer_market_segment, revenue_rank